# Rotated-surface PPM lowering: a two-step sequence

One PPM sequence on THREE patches in a row — `M(Z̄A Z̄B)` through
the corridor between A and B, then `M(Z̄B Z̄C)` through the corridor
between B and C — built two ways: through the kernel API alone
(`lower_ppm` / `apply_ppm_plan` + builder primitives), then through
`RotatedSurfacePPMExperiment`.  Each shows the full-circuit
`detslice-with-ops-svg` (every tick).

Section 3 runs a longer sequence from a compiled program: the
`teleportation_n3` PPM schedule, four PPMs on four patches with their
tiles and corridors.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import contextlib
import io

import stim

import lightstim
from lightstim.ir.builder import CircuitBuilder
from lightstim.ir.qec_system import QECSystem
from lightstim.ir.tracker import SyndromeTracker
from lightstim.qec_code.surface_code.rotated import RotatedSurfaceCode
from lightstim.qec_code.surface_code.rotated.bent_joint_se import (
    se_round_chunk)
from lightstim.qec_code.surface_code.rotated.ppm import (
    RotatedSurfacePatchPlacement, RotatedSurfacePPMRequest,
    apply_ppm_plan, lower_ppm, origin_of)
from lightstim.protocols.rotated_surface_ppm import (
    RotatedSurfacePPMExperiment, RotatedSurfacePPMStep)

# the environment may alias bare lightstim imports elsewhere via a .pth -
# always verify which tree this notebook runs against
print("lightstim from:", lightstim.__file__)


In [ ]:
D = 3          # code distance (rounds = D in every merged window)

px = [RotatedSurfacePatchPlacement('A', origin_of(0, 0, D, seam=True), D, 'X_horizontal'),
      RotatedSurfacePatchPlacement('B', origin_of(2, 0, D, seam=True), D, 'X_horizontal'),
      RotatedSurfacePatchPlacement('C', origin_of(4, 0, D, seam=True), D, 'X_horizontal')]
STATES = {'A': 'Z', 'B': 'Z', 'C': 'Z'}
# two DIFFERENT products through two different corridors
requests = [RotatedSurfacePPMRequest(targets=(('A', 'Z'), ('B', 'Z')), route=((1, 0),)),
            RotatedSurfacePPMRequest(targets=(('B', 'Z'), ('C', 'Z')), route=((3, 0),))]


## 1. Without `RotatedSurfacePPMExperiment`: the kernel API end to end


In [ ]:
system = QECSystem()
for s in px:
    p = RotatedSurfaceCode(distance=s.distance)
    if s.orientation == 'X_horizontal':
        p.transpose_coords()
    system.add_patch(p, name=s.name,
                     offset=(s.origin[0] - 1, s.origin[1] - 1))

tracker = SyndromeTracker(num_qubits=system.num_qubits,
                          expected_num_logicals=system.num_logicals)
builder = CircuitBuilder(tracker=tracker, system_config=system,
                         if_detector=True)
system.register_tracker(tracker)
system.register_builder(builder)
builder.write_coordinates()
owner = system.index_to_owner_map
builder.initialize(
    init_dict={q: STATES[owner[q]] for q in system.data_indices
               if owner.get(q) in STATES},
    n=system.num_qubits)
orient = {s.name: s.orientation for s in px}
domains = {tuple(system.qubit_coords[q]): orient[owner[q]]
           for q in system.data_indices if owner.get(q) in orient}
builder.apply_syndrome_extraction(
    circuit_chunk=se_round_chunk(system, domains=domains), rounds=1)

for i, request in enumerate(requests):
    # Define-by-run: register each coupler only after the baseline has
    # established the logical patches, immediately before that PPM.
    plan = lower_ppm(px, request, system=system)
    apply_ppm_plan(system, plan, f'ppm_{i}')
    cname = f'ppm_{i}'
    builder.activate_coupler(cname)
    cp = system.coupler_patches[cname]
    l2g = system.local_to_global_map[cname]
    corridor_init = {l2g[q]: plan.corridor_init_basis
                     for q in cp.data_indices}
    builder.initialize(init_dict=corridor_init, n=system.num_qubits)
    merged = dict(plan.route_result.layout.domains or {})
    for q in system.data_indices:
        nm = owner.get(q)
        if nm in orient:
            merged.setdefault(tuple(system.qubit_coords[q]), orient[nm])
    builder.apply_syndrome_extraction(
        circuit_chunk=se_round_chunk(system, domains=merged), rounds=D)
    builder.deactivate_coupler(cname)
    builder.apply_data_readout(final_measurements=dict(corridor_init),
                               resolve_absorbed=False)

builder.apply_data_readout(
    final_measurements={q: STATES[owner[q]] for q in system.data_indices
                        if owner.get(q) in STATES})
c_free = builder.circuit
print('qubits:', c_free.num_qubits, '| ticks:', c_free.num_ticks,
      '| detectors:', c_free.num_detectors,
      '| observables:', c_free.num_observables)
det, obs = c_free.compile_detector_sampler(seed=0).sample(
    1024, separate_observables=True)
print('p=0 silent:', not det.any() and not obs.any())
c_free.diagram('detslice-with-ops-svg', tick=range(1, c_free.num_ticks + 1))


## 2. With `RotatedSurfacePPMExperiment`: the same sequence


In [ ]:
exp = RotatedSurfacePPMExperiment(
    px,
    [RotatedSurfacePPMStep([('A', 'Z'), ('B', 'Z')], route=[(1, 0)]),
     RotatedSurfacePPMStep([('B', 'Z'), ('C', 'Z')], route=[(3, 0)])],
    initial_states=STATES, final_measure_states=STATES,
    rounds=D, rounds_init=1)
with contextlib.redirect_stdout(io.StringIO()):
    circuit = exp.build()
print('qubits:', circuit.num_qubits, '| ticks:', circuit.num_ticks,
      '| detectors:', circuit.num_detectors,
      '| observables:', circuit.num_observables)
det, obs = circuit.compile_detector_sampler(seed=0).sample(
    1024, separate_observables=True)
print('p=0 silent:', not det.any() and not obs.any())
circuit.diagram('detslice-with-ops-svg', tick=range(1, circuit.num_ticks + 1))


## 3. A longer sequence: `teleportation_n3`

The patches, the four PPMs and every corridor below are the
`teleportation_n3` example read from its tqec block graph.  Two things
keep it out of `RotatedSurfacePPMExperiment`: the sequence anti-commutes
(PPM 0 measures X̄ on `q0`, PPM 1 measures Z̄), and PPM 0's corridor runs
through the tile where `a2` is born later.  So it is built with the
kernel API, define-by-run: `a2` is added right before PPM 1.  `a0` and
`a1` are kept until the final readout.  The drawing shows PPM 3's merged window.

In [ ]:
TELE = {'q0': ((3, 1), 'X_vertical'), 'a0': ((1, 3), 'X_horizontal'),
        'a1': ((3, 3), 'X_vertical'), 'a2': ((1, 1), 'X_vertical')}
TELE_STATES = {'q0': 'Z', 'a0': 'X', 'a1': 'X', 'a2': 'Z'}   # init = readout letters
tele_px = {nm: RotatedSurfacePatchPlacement(nm, origin_of(*tile, D, seam=True), D, o)
           for nm, (tile, o) in TELE.items()}
tele_requests = [
    RotatedSurfacePPMRequest(targets=(('a0', 'X'), ('q0', 'X')), route=((1, 1), (1, 2), (2, 1))),
    RotatedSurfacePPMRequest(targets=(('a1', 'Z'), ('q0', 'Z')), route=((3, 2),)),
    RotatedSurfacePPMRequest(targets=(('a2', 'X'), ('q0', 'X')), route=((2, 1),)),
    RotatedSurfacePPMRequest(targets=(('a2', 'Z'), ('q0', 'Z')), route=((1, 0), (2, 0), (3, 0))),
]
LATE = {'a2': 1}   # a2 is added right before PPM 1
windows = []       # (first tick, last tick) of each PPM's merged rounds

tsys = QECSystem()
live = []

def add_patch(nm):
    s = tele_px[nm]
    p = RotatedSurfaceCode(distance=s.distance)
    if s.orientation == 'X_horizontal':
        p.transpose_coords()
    tsys.add_patch(p, name=nm, offset=(s.origin[0] - 1, s.origin[1] - 1))
    live.append(nm)

def patch_domains():
    owner = tsys.index_to_owner_map
    return {tuple(tsys.qubit_coords[q]): tele_px[owner[q]].orientation
            for q in tsys.data_indices if owner.get(q) in live}

def init_patches(names):
    owner = tsys.index_to_owner_map
    tbuilder.initialize(init_dict={q: TELE_STATES[owner[q]] for q in tsys.data_indices
                                   if owner.get(q) in names}, n=tsys.num_qubits)
    tbuilder.apply_syndrome_extraction(
        circuit_chunk=se_round_chunk(tsys, domains=patch_domains()), rounds=1)

for nm in TELE:
    if nm not in LATE:
        add_patch(nm)
ttracker = SyndromeTracker(num_qubits=tsys.num_qubits, expected_num_logicals=tsys.num_logicals)
tbuilder = CircuitBuilder(tracker=ttracker, system_config=tsys, if_detector=True)
tsys.register_tracker(ttracker)
tsys.register_builder(tbuilder)
tbuilder.write_coordinates()
with contextlib.redirect_stdout(io.StringIO()):
    init_patches(set(live))
    for i, request in enumerate(tele_requests):
        for nm, before in LATE.items():
            if before == i:                      # define-by-run birth
                add_patch(nm)
                tbuilder.write_coordinates()
                init_patches({nm})
        plan = lower_ppm([tele_px[nm] for nm in live], request, system=tsys)
        cname = f'tele_ppm_{i}'
        apply_ppm_plan(tsys, plan, cname)
        tbuilder.activate_coupler(cname)
        cp = tsys.coupler_patches[cname]
        l2g = tsys.local_to_global_map[cname]
        corridor_init = {l2g[q]: plan.corridor_init_basis for q in cp.data_indices}
        tbuilder.initialize(init_dict=corridor_init, n=tsys.num_qubits)
        t0 = tbuilder.circuit.num_ticks
        merged = dict(plan.route_result.layout.domains or {})
        for coord, orientation in patch_domains().items():
            merged.setdefault(coord, orientation)
        tbuilder.apply_syndrome_extraction(
            circuit_chunk=se_round_chunk(tsys, domains=merged), rounds=D)
        windows.append((t0 + 1, tbuilder.circuit.num_ticks))
        tbuilder.deactivate_coupler(cname)
        tbuilder.apply_data_readout(final_measurements=dict(corridor_init),
                                    resolve_absorbed=False)
    owner = tsys.index_to_owner_map
    tbuilder.apply_data_readout(
        final_measurements={q: TELE_STATES[owner[q]] for q in tsys.data_indices
                            if owner.get(q) in live and q in tsys.active_qubit_indices})
c_tele = tbuilder.circuit
print('qubits:', c_tele.num_qubits, '| ticks:', c_tele.num_ticks,
      '| detectors:', c_tele.num_detectors,
      '| observables:', c_tele.num_observables)
det, obs = c_tele.compile_detector_sampler(seed=0).sample(
    1024, separate_observables=True)
print('p=0 silent:', not det.any() and not obs.any())
first, last = windows[3]
c_tele.diagram('detslice-with-ops-svg', tick=range(first, last + 1))